# Experiment Tracking with TensorBoard

Without experiment tracking, comparing two runs means squinting at `print()` output from different terminal sessions.

TensorBoard ships with PyTorch — zero extra install. It records metrics, images, model graphs, and hyperparameters to disk, then displays them in an interactive UI.

**What we'll log:**
- Scalar metrics: train/val loss and accuracy per epoch
- Images: training batch samples
- Model graph: the full computational graph
- Hyperparameters + final metric: compare runs in one table

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

## 1. `SummaryWriter` basics

`SummaryWriter(log_dir)` creates a writer that saves events to `log_dir`.  
Use a **unique subdirectory per run** so runs don't overwrite each other.

In [ ]:
from datetime import datetime

def make_run_name(lr, batch_size, model_name):
    """Unique, readable run name for TensorBoard log directory."""
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    return f'runs/{model_name}/lr{lr}_bs{batch_size}_{ts}'

# Example
print(make_run_name(0.001, 64, 'cnn_mnist'))

## 2. Data

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_ds = datasets.MNIST('./data', train=True,  download=True, transform=transform)
test_ds  = datasets.MNIST('./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=64)

## 3. Model

In [ ]:
import torch.nn.functional as F

class TinyCNN(nn.Module):
    def __init__(self, hidden_size=64):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, 3)
        self.conv2 = nn.Conv2d(16, 32, 3)
        self.pool  = nn.MaxPool2d(2)
        self.fc1   = nn.Linear(32 * 5 * 5, hidden_size)
        self.fc2   = nn.Linear(hidden_size, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = F.relu(self.fc1(x.flatten(1)))
        return self.fc2(x)

## 4. Training loop with TensorBoard logging

In [ ]:
def train_with_tb(lr=1e-3, hidden_size=64, epochs=5):
    model     = TinyCNN(hidden_size=hidden_size).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    run_name  = make_run_name(lr, 64, f'tinycnn_h{hidden_size}')
    writer    = SummaryWriter(run_name)

    # Log model graph (needs a sample input)
    sample_input = torch.zeros(1, 1, 28, 28).to(DEVICE)
    writer.add_graph(model, sample_input)

    # Log a batch of training images
    imgs, labels = next(iter(train_loader))
    writer.add_images('train/sample_batch', imgs[:16], global_step=0)

    for epoch in range(epochs):
        # --- train ---
        model.train()
        train_loss = 0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        train_loss /= len(train_loader)

        # --- eval ---
        model.eval()
        val_loss = correct = total = 0
        with torch.no_grad():
            for xb, yb in test_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                logits  = model(xb)
                val_loss += criterion(logits, yb).item()
                correct  += (logits.argmax(1) == yb).sum().item()
                total    += yb.size(0)
        val_loss /= len(test_loader)
        val_acc   = correct / total

        # --- log scalars ---
        writer.add_scalar('Loss/train', train_loss, epoch)
        writer.add_scalar('Loss/val',   val_loss,   epoch)
        writer.add_scalar('Accuracy/val', val_acc,  epoch)

        # Log learning rate
        writer.add_scalar('LR', optimizer.param_groups[0]['lr'], epoch)

        print(f'[{run_name.split("/")[2]}] epoch {epoch+1}  '
              f'train_loss={train_loss:.4f}  val_acc={val_acc:.4f}')

    # Log hyperparameters + final metric (shows in TensorBoard HPARAMS tab)
    writer.add_hparams(
        {'lr': lr, 'hidden_size': hidden_size, 'epochs': epochs},
        {'hparam/val_acc': val_acc, 'hparam/val_loss': val_loss}
    )

    writer.close()
    return val_acc

In [ ]:
# Run experiment 1 — baseline
acc1 = train_with_tb(lr=1e-3, hidden_size=64, epochs=5)
print(f'Run 1 val_acc: {acc1:.4f}')

In [ ]:
# Run experiment 2 — higher LR
acc2 = train_with_tb(lr=1e-2, hidden_size=64, epochs=5)
print(f'Run 2 val_acc: {acc2:.4f}')

In [ ]:
# Run experiment 3 — larger hidden size
acc3 = train_with_tb(lr=1e-3, hidden_size=128, epochs=5)
print(f'Run 3 val_acc: {acc3:.4f}')

## 5. Launch TensorBoard

In [ ]:
# In a notebook:
%load_ext tensorboard
%tensorboard --logdir runs

In [ ]:
# Or from terminal:
# tensorboard --logdir runs
# Then open http://localhost:6006

## 6. What each TensorBoard tab shows

| Tab | What to look for |
|---|---|
| **SCALARS** | Loss curves — train/val diverging = overfitting |
| **IMAGES** | Training batch — sanity check your data pipeline |
| **GRAPHS** | Model architecture — verify the layers are connected as expected |
| **HPARAMS** | Table of all runs with final metrics — find the best config |

## 7. `add_scalar` vs `add_scalars`

In [ ]:
# add_scalars — plot multiple lines on the same chart
writer = SummaryWriter('runs/demo')

for step in range(10):
    writer.add_scalars('Loss/comparison', {
        'train': 1.0 / (step + 1),
        'val':   1.2 / (step + 1)
    }, global_step=step)

writer.close()
print('Demo run written to runs/demo')